<a href="https://colab.research.google.com/github/chelsietao/LLM-Refusal-Mechanism-Exploration/blob/rebuild/v3/LLM_Refusal_Mechanism_Exploration_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Refusal Mechanism Exploration — v3

What v3 adds over v2 (all logic lives in `src/refusal_lab/` and is unit-tested on CPU):

1. **Data file**: 220 categorized prompt pairs in `data/prompts_v3.jsonl`, stratified split (≈132 / 44 / 44 pairs).
2. **All-layer, all-position ablation** (Arditi et al. 2024) next to the v1/v2 single-layer ablation.
3. **Steering** (activation addition on harmless prompts) and **shuffled-label / random controls** (norm-matched).
4. **Weight orthogonalization** and a **latency benchmark** (baseline vs hook vs edited weights).

Fixes carried in v3: activations are captured with hooks on decoder layers (the last `hidden_states` entry is *post final norm*), and the chat template's BOS is no longer duplicated.

> **Safety:** research use only. Generated text is never stored; logs hold labels, hashes and lengths.
> **Not run yet:** this notebook has been written and its library tested on a tiny CPU model, but no GPU results exist until you run it.

In [ ]:
import subprocess, sys
from pathlib import Path
REPO_URL = "https://github.com/chelsietao/LLM-Refusal-Mechanism-Exploration.git"
REPO_BRANCH = "rebuild/v3"
REPO_DIR = Path("/content/LLM-Refusal-Mechanism-Exploration")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
%pip install -q "transformers>=4.44,<5" "accelerate>=0.33" "huggingface_hub>=0.24" pandas scikit-learn matplotlib seaborn tqdm
sys.path.insert(0, str(REPO_DIR / "src"))

## 1. Configuration

In [ ]:
import json, gc, random
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from contextlib import nullcontext

SEED = 0
MODEL_ID = "google/gemma-2-2b-it"                # main study model (gated: accept terms on Hugging Face)
EXACT_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"    # part 4: weight edit is exactly equivalent on this architecture
QUICK_MODE = True      # True: every 2nd layer, 32 new tokens. Set False for the real run.
ARTIFACT_DIR = REPO_DIR / "artifacts" / "v3"; ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_grad_enabled(False)

from refusal_lab.data import load_pairs, split_pairs
from refusal_lab.evaluate import EvalConfig, fit_directions, select_layer, evaluate_locked_test, summarize
from refusal_lab.interventions import ablate_everywhere
from refusal_lab.orthogonalize import supports_exact_orthogonalization
from refusal_lab.bench import benchmark_all, time_condition, compare_logits
from transformers import AutoModelForCausalLM, AutoTokenizer

def load(model_id):
    try:
        from google.colab import userdata; token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
    if not token:
        from huggingface_hub import notebook_login; notebook_login()
    if not torch.cuda.is_available():
        raise RuntimeError("GPU runtime required (Runtime -> Change runtime type -> T4 GPU).")
    dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    tok = AutoTokenizer.from_pretrained(model_id, token=token); tok.padding_side = "left"
    if tok.pad_token_id is None: tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(model_id, token=token, torch_dtype=dtype, device_map="auto").eval()
    return mdl, tok

cfg = EvalConfig(seed=SEED, batch_size=8, max_new_tokens=32 if QUICK_MODE else 64)
pairs = load_pairs(REPO_DIR / "data" / "prompts_v3.jsonl")
splits = split_pairs(pairs, seed=SEED)
print({k: len(v) for k, v in splits.items()}, "pairs;", len({p['category'] for p in pairs}), "categories")

## 2. Main study: load the model, fit directions on `direction_train` only

In [ ]:
model, tokenizer = load(MODEL_ID)
L = len(model.model.layers)
cfg.candidate_layers = list(range(0, L, 2)) if QUICK_MODE else None
dirs = fit_directions(model, tokenizer, splits["direction_train"], cfg)
print("layers:", L, "| hidden:", dirs.unit.shape[-1])
plt.figure(figsize=(10, 3.5)); plt.plot(dirs.raw.norm(dim=-1).numpy(), marker="o")
plt.title("Difference-of-means norm per layer (direction_train)"); plt.xlabel("layer"); plt.show()

## 3. Validation-only layer selection (all-layer, all-position ablation)

For each candidate source layer `l`, the direction `r_l` is projected out of the token embedding and every attention/MLP output at every position. The `layer_validation` split picks `l`.

In [ ]:
sweep = select_layer(model, tokenizer, splits["layer_validation"], dirs, cfg)
sweep.to_csv(ARTIFACT_DIR / "validation_sweep_all_layer_ablation.csv", index=False)
best_layer = int(sweep.iloc[0]["layer"]); print("selected source layer:", best_layer)
display(sweep.head(10))
s = sweep.sort_values("layer")
plt.figure(figsize=(10, 3.5)); plt.plot(s["layer"], s["ablated_refusal"], marker="o", label="all-position ablation")
plt.axhline(s["baseline_refusal"].iloc[0], ls="--", c="g", label="baseline"); plt.axvline(best_layer, c="r", alpha=.4)
plt.ylim(-.05, 1.05); plt.legend(); plt.xlabel("source layer of direction"); plt.ylabel("validation harmful refusal"); plt.show()

## 4. Locked-test evaluation: ablation, steering, and controls

Conditions: baseline · learned / shuffled-label / random ablation (all positions) · learned single-layer ablation (v2 behaviour) · steering on **harmless** prompts with learned / shuffled / random vectors, all **norm-matched** to the learned vector at the selected layer.

In [ ]:
log = evaluate_locked_test(model, tokenizer, splits["locked_test"], dirs, best_layer, cfg)
log.to_csv(ARTIFACT_DIR / "locked_test_log_no_text.csv", index=False)
summary = summarize(log); summary.to_csv(ARTIFACT_DIR / "locked_test_summary.csv", index=False)
display(summary.sort_values(["intent", "condition", "coeff"]))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
h = summary[(summary.intent == "harmful") & ~summary.condition.str.startswith("steer")]
ax[0].bar(h.condition, h.refusal_rate, yerr=[h.refusal_rate - h.ci95_low, h.ci95_high - h.refusal_rate], capsize=4)
ax[0].set_title("Harmful prompts: refusal rate (95% bootstrap CI)"); ax[0].tick_params(axis="x", rotation=30)
st = summary[summary.condition.str.startswith("steer")]
for tag, g in st.groupby("condition"):
    ax[1].errorbar(g.coeff, g.refusal_rate, yerr=[g.refusal_rate - g.ci95_low, g.ci95_high - g.refusal_rate], marker="o", capsize=3, label=tag)
b = summary[(summary.intent == "harmless") & (summary.condition == "baseline")].refusal_rate.iloc[0]
ax[1].axhline(b, ls="--", c="gray", label="harmless baseline"); ax[1].set_title("Harmless prompts + steering"); ax[1].set_xlabel("coefficient (x learned norm)"); ax[1].legend()
plt.tight_layout(); plt.savefig(ARTIFACT_DIR / "locked_test.png", dpi=160); plt.show()

## 5. Inference: hooks vs. baked-in weights

Weight orthogonalization `W <- (I - r r^T) W` is *exactly* the all-position hook ablation only if nothing element-wise sits between the writing matrices and the residual add. **Gemma-2 applies a scaled RMSNorm to attention/MLP outputs first, so the edit is only approximate there** (the library refuses by default; verified in `tests/test_interventions.py`). Part 4 therefore uses a Llama-style model.

In [ ]:
print("Gemma-2 exact weight edit possible:", supports_exact_orthogonalization(model))
del model; gc.collect(); torch.cuda.empty_cache()
model, tokenizer = load(EXACT_MODEL_ID)
assert supports_exact_orthogonalization(model)
L2 = len(model.model.layers)
cfg2 = EvalConfig(seed=SEED, batch_size=8, max_new_tokens=32)
dirs2 = fit_directions(model, tokenizer, splits["direction_train"], cfg2)
layer2 = L2 // 2     # latency does not depend on direction values; fidelity is checked with this real direction
direction = dirs2.unit[layer2]
bench_prompts = [p["harmless"] for p in splits["locked_test"]][:8]
fidelity = compare_logits(model, tokenizer, bench_prompts, direction)
print("fidelity (hook vs edited weights):", fidelity)

In [ ]:
res = benchmark_all(model, tokenizer, bench_prompts, direction, new_tokens=64, batch_size=8, warmup=2, repeats=7)
bench = pd.DataFrame(res).T
bench["slowdown_vs_baseline"] = bench["median_total_s"] / bench.loc["baseline", "median_total_s"]
display(bench)
bench.to_csv(ARTIFACT_DIR / "latency_benchmark.csv")
(ARTIFACT_DIR / "bench_meta.json").write_text(json.dumps({
    "model": EXACT_MODEL_ID, "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
    "fidelity": fidelity, "direction_layer": layer2}, indent=2))

## 6. Interpretation checklist

- Learned all-position ablation should cut harmful refusal clearly more than the shuffled and random controls (read the CIs, not just the bars).
- Steering should raise harmless refusal for the learned vector but not for norm-matched controls.
- Harmless over-refusal under ablation should stay near baseline.
- `fidelity` should show ~0 KL and 100% argmax agreement; if not, the equivalence assumption failed.
- Timings are for one GPU/runtime; report GPU name, batch size and repeats with any number you quote.
- Still open: a judge-model or human-audited refusal scorer, capability retention (perplexity / benchmarks), more models.